# CMG Valuation Walkthrough

This notebook is a thin, narrated layer over the real logic in `src/` — it doesn't reimplement anything, it just calls the same modules `main.py` uses and shows the intermediate steps. Run `python main.py` from the repo root first (or run all cells here) to (re)generate `outputs/`.

See the [README](../README.md) for the full methodology and assumptions writeup.

In [ ]:
import sys
sys.path.append('..')

import yaml
import pandas as pd

from src.data.fetch import fetch_company_snapshot, fetch_comp_snapshots, fetch_risk_free_rate
from src.dcf.projections import build_fcf_projection
from src.dcf.wacc import compute_wacc
from src.dcf.terminal_value import gordon_growth_terminal_value, enterprise_value
from src.dcf.sensitivity import sensitivity_table
from src.comps.peer_data import build_comps_table, peer_summary_stats
from src.comps.implied_valuation import implied_valuation_from_comps

assumptions = yaml.safe_load(open('../config/assumptions.yaml'))
assumptions['target']

## 1. Pull data (cached snapshot if available)

In [ ]:
target = fetch_company_snapshot(assumptions['target']['ticker'])
target

## 2. Revenue and free cash flow projection

In [ ]:
projection = build_fcf_projection(target.revenue, assumptions['target']['base_fiscal_year'], assumptions)
projection

## 3. Cost of capital (WACC)

CMG has no traditional bonds/bank debt — only capitalized operating leases. WACC is set equal to the CAPM cost of equity. See the README for why.

In [ ]:
risk_free_rate = assumptions['wacc'].get('risk_free_rate') or fetch_risk_free_rate()
wacc = compute_wacc(risk_free_rate, target.beta, assumptions['wacc']['equity_risk_premium'])
print(f'Risk-free rate: {risk_free_rate:.2%}')
print(f'Beta: {target.beta:.2f}')
print(f'WACC: {wacc:.2%}')

## 4. Terminal value, enterprise value, implied share price

In [ ]:
fcfs = projection['unlevered_fcf'].tolist()
terminal_growth = assumptions['dcf']['terminal_growth_rate']
tv = gordon_growth_terminal_value(fcfs[-1], wacc, terminal_growth)
ev_result = enterprise_value(fcfs, tv, wacc)

net_debt = target.total_debt - target.cash
equity_value = ev_result['enterprise_value'] - net_debt
dcf_price = equity_value / target.shares_diluted

print(f"Enterprise value: ${ev_result['enterprise_value']/1e9:.2f}B")
print(f'Net debt: ${net_debt/1e9:.2f}B')
print(f'Equity value: ${equity_value/1e9:.2f}B')
print(f'DCF implied share price: ${dcf_price:.2f}')
print(f'Current market price: ${target.price:.2f}')

## 5. Sensitivity: WACC vs. terminal growth

In [ ]:
sens = sensitivity_table(
    fcfs,
    assumptions['dcf']['sensitivity']['wacc_range'],
    assumptions['dcf']['sensitivity']['terminal_growth_range'],
    net_debt,
    target.shares_diluted,
)
sens.round(2)

## 6. Comparable companies analysis

In [ ]:
peers = fetch_comp_snapshots(assumptions['comps']['tickers'])
comps_df = build_comps_table(peers)
comps_df

In [ ]:
peer_stats = peer_summary_stats(comps_df)
peer_stats

In [ ]:
implied_df = implied_valuation_from_comps(target, peer_stats)
implied_df

## 7. Summary

See `outputs/figures/valuation_summary.png` for the DCF-vs-comps-vs-market chart, generated by `main.py`. The full writeup of what these numbers mean and their limitations is in the [README](../README.md).